# DOCENTE · Preparar la copia de respaldo de los datos

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/modelacion-sismica-tsunami-espam/blob/main/cuadernos/DOCENTE_preparar_respaldo.ipynb)

Descarga, para los 11 eventos de la guía, los datos que usa el cuaderno del Capítulo 1 y los deja listos para subir a `datos/respaldo/` del repositorio. Si el USGS o Global CMT no responden el día de la práctica, el cuaderno de los estudiantes lee esta copia.

**Uso (una sola vez, unos 5 minutos):**
1. Ejecuten todas las celdas en Colab.
2. Se descarga `respaldo.zip`.
3. En GitHub: *Add file → Upload files*, arrastren el **contenido** de la carpeta `respaldo/` descomprimida dentro de `datos/respaldo/` y confirmen (*Commit changes*).

Al final, la tabla resume qué se obtuvo de cada evento; revísenla antes de subir.

In [ ]:
import io, os, json, math, gzip, shutil, datetime as dt
import numpy as np, pandas as pd, requests

FDSN = "https://earthquake.usgs.gov/fdsnws/event/1/query"
EVENTOS = {
    "Pedernales 2016": dict(fecha="2016-04-16", lat=0.38,   lon=-79.92,  Mw=7.8),
    "Maule 2010":      dict(fecha="2010-02-27", lat=-36.12, lon=-72.90,  Mw=8.8),
    "Illapel 2015":    dict(fecha="2015-09-16", lat=-31.57, lon=-71.67,  Mw=8.3),
    "Iquique 2014":    dict(fecha="2014-04-01", lat=-19.61, lon=-70.77,  Mw=8.2),
    "Pisco 2007":      dict(fecha="2007-08-15", lat=-13.39, lon=-76.60,  Mw=8.0),
    "Camaná 2001":     dict(fecha="2001-06-23", lat=-16.26, lon=-73.64,  Mw=8.4),
    "Tohoku 2011":     dict(fecha="2011-03-11", lat=38.30,  lon=142.37,  Mw=9.1),
    "Nicaragua 1992":  dict(fecha="1992-09-02", lat=11.74,  lon=-87.34,  Mw=7.7),
    "Mentawai 2010":   dict(fecha="2010-10-25", lat=-3.49,  lon=100.08,  Mw=7.8),
    "Samoa 2009":      dict(fecha="2009-09-29", lat=-15.49, lon=-172.10, Mw=8.1),
    "Océano Índico 2012": dict(fecha="2012-04-11", lat=2.33, lon=93.06,  Mw=8.6),
}
def slug(t):
    t = t.lower()
    for a, b in zip("áéíóúñ ", "aeioun_"):
        t = t.replace(a, b)
    return t
def get(url, binario=False, timeout=180):
    try:
        r = requests.get(url, timeout=timeout, headers={"User-Agent": "IRD505-ESPAM-docencia"}); r.raise_for_status()
        return r.content if binario else r.text
    except Exception as e:
        print("  falló:", url[:100], type(e).__name__); return None

# Catálogo completo de Global CMT (una sola descarga)
NDK = None
for a in range(dt.date.today().year % 100, 19, -1):
    for pr in ("https", "http"):
        d = get(f"{pr}://www.ldeo.columbia.edu/~gcmt/projects/CMT/catalog/jan76_dec{a:02d}.ndk.gz", binario=True)
        if d:
            NDK = gzip.decompress(d).decode("latin-1").splitlines(); print("Global CMT: jan76_dec%02d" % a); break
    if NDK: break

In [ ]:
os.makedirs("respaldo", exist_ok=True)
resumen = []
for nombre, ev in EVENTOS.items():
    print("·", nombre)
    carpeta = os.path.join("respaldo", slug(nombre)); os.makedirs(carpeta, exist_ok=True)
    fila = dict(evento=nombre)
    t0 = (pd.Timestamp(ev["fecha"]) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    t1 = (pd.Timestamp(ev["fecha"]) + pd.Timedelta(days=2)).strftime("%Y-%m-%d")
    txt = get(f"{FDSN}?format=geojson&starttime={t0}&endtime={t1}&latitude={ev['lat']}&longitude={ev['lon']}"
              f"&maxradius=3&minmagnitude={ev['Mw'] - 0.7:.1f}&orderby=magnitude&limit=1")
    feats = json.loads(txt)["features"] if txt else []
    if not feats:
        resumen.append(dict(fila, usgs="NO")); continue
    det_txt = get(f"{FDSN}?eventid={feats[0]['id']}&format=geojson")
    det = json.loads(det_txt)
    open(os.path.join(carpeta, "usgs_detalle.json"), "w").write(det_txt)
    p = det["properties"]; lon, lat, z = det["geometry"]["coordinates"]
    fila.update(usgs=det["id"], Mw=p["mag"], tensor="moment-tensor" in p.get("products", {}),
                falla_finita="finite-fault" in p.get("products", {}))
    t_ref = pd.to_datetime(p["time"], unit="ms")
    # Réplicas: radio y magnitud mínima holgados; el cuaderno filtra después
    L = 10 ** (-2.477 + 0.585 * p["mag"])
    rep = get(f"{FDSN}?format=csv&starttime={t_ref:%Y-%m-%dT%H:%M:%S}&endtime={t_ref + pd.Timedelta(days=30):%Y-%m-%dT%H:%M:%S}"
              f"&latitude={lat}&longitude={lon}&maxradiuskm={max(250, 1.6 * L):.0f}&minmagnitude=3.5&orderby=time-asc")
    if rep:
        open(os.path.join(carpeta, "replicas.csv"), "w").write(rep)
        fila["replicas"] = len(pd.read_csv(io.StringIO(rep)))
    # Global CMT: bloque del sismo más cercano en tiempo
    if NDK:
        mejor, dmin = None, 1e9
        for i in range(0, len(NDK) - 4, 5):
            try:
                t = pd.Timestamp(NDK[i][5:15].replace("/", "-") + " " + NDK[i][16:26].strip())
            except Exception:
                continue
            dd = abs((t - t_ref).total_seconds())
            if dd < dmin:
                dmin, mejor = dd, "\n".join(NDK[i:i + 5]) + "\n"
        if mejor and dmin < 180:
            open(os.path.join(carpeta, "gcmt.ndk"), "w").write(mejor)
            fila["gcmt"] = mejor.splitlines()[1][:16].strip()
    resumen.append(fila)

display(pd.DataFrame(resumen))
shutil.make_archive("respaldo", "zip", ".", "respaldo")
try:
    from google.colab import files
    files.download("respaldo.zip")
except ImportError:
    print("respaldo.zip creado")